<a href="https://colab.research.google.com/github/ACE6233/Machine-Learning/blob/main/ylfoo/T2530_ARA6123/ARA6123_Lab_Test_1(T2530)_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ARA6123/AAT6123/ACE6313 Lab Test 1 (1 hour)

#### Name:
#### Student ID:
#### Date:

### Description
#### The Steel Industry Energy Consumption dataset provides 35,040 real-world observations collected from a steel manufacturing facility in South Korea to support research in smart industry energy analytics. It contains detailed measurements of electricity usage (in kWh), power factor, CO₂ emissions, temporal attributes, and operational load categories. The dataset captures both continuous and categorical features, offering a foundation for modelling energy consumption patterns and understanding the operational behaviour of industrial equipment across different time periods.

### Tasks
In this lab test, you are required to complete the following tasks:  
1. Load the dataset from a csv file to a Pandas DataFrame.
2. Print the first 5 rows of the dataset.
3. Check for missing values.
4. Impute the missing values if there are any (skip this step if there is no missing value).
5. Perform categorical encoding (either ordinal encoding or one-hot encoding) to the categorical columns. To check the unique values of Column **ABC**, you can use **print(df['ABC'].unique())**
6. Separate the dataset to features (X) and target (y).
7. Use 80% of the data samples for training and the remaining data samples for testing.
8. Apply MinMaxScaling to the features.
9. Use spot-checking technique with 5-fold cross validation to quickly evaluate the performance of Linear Regression, kNN Regression, Random Forest Regression and Multilayer Perceptron Regression to predict the electricity usage based on the scaled features
10. Train a model using the best performing algorithm obtained in Step (9) with the entire training set and then evaluate it using the testing set.

*Use **random_state=42** for reproducibility and the results should be shown up to **3** decimal places*

In [ ]:
# Load the libraries
import pandas as pd
from sklearn.model_selection import train_test_split as split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ylfoo/T2530_ARA6123/main/steel_industry_energy_data.csv')

In [ ]:
# df = pd.read_csv('steel_industry_energy_data.csv')

In [ ]:
df.head()

,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Load_Type
0,3.17,0.0,73.21,100.0,900.0,Weekday,Light_Load
1,4.00,0.0,66.77,100.0,1800.0,Weekday,Light_Load
2,3.24,0.0,70.28,100.0,2700.0,Weekday,Light_Load
3,3.31,0.0,68.09,100.0,3600.0,Weekday,Light_Load
4,3.82,0.0,64.72,100.0,4500.0,Weekday,Light_Load


In [ ]:
df.isna().sum()

Usage_kWh                        0
CO2(tCO2)                       11
Lagging_Current_Power_Factor     0
Leading_Current_Power_Factor     0
NSM                              9
WeekStatus                       0
Load_Type                        0
dtype: int64

In [ ]:
df =  df.fillna({'CO2(tCO2)': df['CO2(tCO2)'].median(), 'NSM': df['NSM'].median()})
df.isna().sum()

Usage_kWh                       0
CO2(tCO2)                       0
Lagging_Current_Power_Factor    0
Leading_Current_Power_Factor    0
NSM                             0
WeekStatus                      0
Load_Type                       0
dtype: int64

In [ ]:
print(df['Load_Type'].unique())

['Light_Load' 'Medium_Load' 'Maximum_Load']


In [ ]:
print(df['WeekStatus'].unique())

['Weekday' 'Weekend']


In [ ]:
mapping = {'Light_Load':1, 'Medium_Load':2, 'Maximum_Load':3}
df['Load_Type'] = df['Load_Type'].map(mapping)

In [ ]:
df2 = pd.get_dummies(df, columns=['WeekStatus'])
df2.head()

,Usage_kWh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,Load_Type,WeekStatus_Weekday,WeekStatus_Weekend
0,3.17,0.0,73.21,100.0,900.0,1,True,False
1,4.00,0.0,66.77,100.0,1800.0,1,True,False
2,3.24,0.0,70.28,100.0,2700.0,1,True,False
3,3.31,0.0,68.09,100.0,3600.0,1,True,False
4,3.82,0.0,64.72,100.0,4500.0,1,True,False


In [ ]:
X = df2.drop(columns=['Usage_kWh'])
y = df2['Usage_kWh']

In [ ]:
X_train, X_test, y_train, y_test = split(X, y, test_size=0.2, random_state=42)

In [ ]:
scl = MinMaxScaler()
Xs_train = scl.fit_transform(X_train)
Xs_test = scl.transform(X_test)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {}
models['lnr'] = LinearRegression()
models['knn'] = KNeighborsRegressor()
models['rfr'] = RandomForestRegressor()
models['mlp'] = MLPRegressor()

for n in models:
    scores = cross_val_score(models[n], Xs_train, y_train, cv=kf, n_jobs=-1)
    print(f'{n}: {scores.mean():.3f} +/- {scores.std():.3f}')

lnr: 0.975 +/- 0.002
knn: 0.986 +/- 0.000
rfr: 0.987 +/- 0.000
mlp: 0.976 +/- 0.003


In [ ]:
best_model = RandomForestRegressor(random_state=42).fit(Xs_train, y_train)
print(f'Best model: {best_model.score(Xs_test, y_test):.3f}')

Best model: 0.987
